In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
import json

load_dotenv()
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

In [4]:
from pydantic import BaseModel
from typing import List

class Questions(BaseModel):
    questions: List[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

def llm_structured(instructions, user_prompt, output_type, model="openai/gpt-oss-120b"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = client.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type
    )
    return response.choices[0].message.parsed, response.usage

target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

sample_gt = []
usages = []

for doc in documents:
    if doc["filename"] not in target_files:
        continue
    user_prompt = json.dumps(doc)
    parsed, usage = llm_structured(data_gen_instructions, user_prompt, Questions)
    usages.append(usage)
    for q in parsed.questions:
        sample_gt.append({"question": q, "filename": doc["filename"]})

avg_input_tokens = sum(u.prompt_tokens for u in usages) / len(usages)
avg_input_tokens

1482.0

In [5]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
df_ground_truth = pd.read_csv(f"{PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

360

In [7]:
from gitsource import chunk_documents
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

# --- Text search ---
index = Index(
    text_fields=['text'],
    keyword_fields=['filename']
)
index.fit(chunks)

def text_search(query, num_results=5):
    boost_dict = {'text': 1.0}
    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

# --- Vector search ---
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [c['text'] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(np.array(embeddings), chunks)

def vector_search(query, num_results=5):
    query_vector = model.encode(query)
    return vindex.search(query_vector, num_results=num_results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyError: 'text'

In [ ]:
q = ground_truth[0]["question"]

text_search(q)[0]["filename"]   # Q2
vector_search(q)[0]["filename"] # Q3

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
def compute_relevance(search_function, q):
    results = search_function(q["question"])
    return [1 if d["filename"] == q["filename"] else 0 for d in results]

def hit_rate(relevance_total):
    return sum(1 if any(r) else 0 for r in relevance_total) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for r in relevance_total:
        for rank, val in enumerate(r):
            if val:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = [compute_relevance(search_function, q) for q in ground_truth]
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

In [ ]:
evaluate(ground_truth, text_search)      # Q4 -> hit_rate
evaluate(ground_truth, vector_search)    # Q5 -> mrr

In [ ]:
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda q, k=k: hybrid_search(q, k=k))
    print(k, result["mrr"])